# 🎥 YouTube Video Summarization using BART

## Project Overview

This notebook demonstrates how to summarize a YouTube video using Hugging Face's BART model. The workflow includes:

- Extracting the YouTube transcript.
- Counting input tokens.
- Applying chunking for long transcripts.
- Generating a summary using BART.

In [1]:
pip install youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 25.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi

In [3]:
def extract_video_id(url: str) -> str:
    """
    Extract the YouTube video ID from a URL.
    Raises ValueError if no 'v' parameter is found.
    """
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    video_ids = qs.get('v')

    if not video_ids:
        raise ValueError(f"No video id found in URL: {url}")
    return video_ids[0]

if __name__ == "__main__":
    url = "https://www.youtube.com/watch?v=_hdUddANh_o"

    video_id = extract_video_id(url)

    api = YouTubeTranscriptApi()
    fetched = api.fetch(video_id, languages=['en']) # returns a FetchedTranscript

    text = "\n".join(snippet.text for snippet in fetched)

print(text)

Most people who try to learn machine
learning quit not because it's too hard,
but because they waste months on the
wrong things. They binge watch lecture
series, memorize math that they'll never
use, and never actually build anything.
Now, I've seen it hundreds of times, and
in this video, I'm going to give you the
exact learning path that actually works
step by step. What to learn, what to
skip, and how to learn it in a way where
you're not spinning your wheels for 6
months with nothing to show for it. And
it's a shame when that happens because
ML engineers are some of the highest
paid people in tech right now. And we're
talking, you know, 150 to 200k plus just
for starting roles. So with that said,
let's dive in and let me discuss how to
learn machine learning like a genius so
that you don't give up. Now I want to
start by discussing the trap that almost
everybody falls into when they're trying
to get into machine learning. Now that's
trying to learn everything before they
build anyt

In [4]:
!pip install "transformers<5" torch sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 113.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 112.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.8 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This

In [5]:
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

2026-06-29 22:25:32.648883: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782771932.866211      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782771932.926280      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782771933.427192      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782771933.427232      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782771933.427235      58 computation_placer.cc:177] computation placer alr

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [6]:
print(len(text))
print(len(text.split()))

18931
3400


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

inputs = tokenizer(text)

print(len(inputs["input_ids"]))

4731


In [8]:
from nltk.tokenize import sent_tokenize

In [9]:
import nltk
nltk.download('punkt_tab', quiet=True)

max_tokens = 1024

sentences = sent_tokenize(text)

chunks = []
current_chunk = ""

for sentence in sentences:
    test_chunk = current_chunk + sentence + ". "

    num_tokens = len(tokenizer.encode(test_chunk))

    if num_tokens <= max_tokens:
        current_chunk = test_chunk
    else:
        chunks.append(current_chunk.strip())
        current_chunk = sentence + ". "

if current_chunk:
    chunks.append(current_chunk.strip())

In [16]:
summaries = []

for chunk in chunks:
    summary = summarizer(
        chunk,
        max_length=150,
        min_length=50,
        do_sample=False
    )

    summaries.append(summary[0]["summary_text"])

In [17]:
final_summary = "\n".join(summaries)

print(final_summary)

Most people who try to learn machine learning quit not because it's too hard, but because they waste months on the wrong things. The fix here is to learn just enough theory to understand what's happening, but then immediately start applying this and actually building projects. You learn way more when you're doing hands-on coding, you're solving problems, and dealing with challenges.
You don't need to be able to derive all of these algorithms or write them from scratch. You just need to have looked at them before, understand at a fundamental level kind of why they work or what they're for. It's only if you're going into super-deep research or really advanced machine learning that you'll need to actually be good and solid in these topics.
Data Camp is a great way to learn machine learning without wasting months stitching together random tutorials. PyTorch is becoming more and more popular as the production standard for machine learning. neural networks are going to be feed forward neural

In [18]:
print("========== Original Transcript ==========\n")
print(text[:1500])   
print("\n========== Generated Summary ==========\n")
print(final_summary)

========== Original Transcript ==========

Most people who try to learn machine
learning quit not because it's too hard,
but because they waste months on the
wrong things. They binge watch lecture
series, memorize math that they'll never
use, and never actually build anything.
Now, I've seen it hundreds of times, and
in this video, I'm going to give you the
exact learning path that actually works
step by step. What to learn, what to
skip, and how to learn it in a way where
you're not spinning your wheels for 6
months with nothing to show for it. And
it's a shame when that happens because
ML engineers are some of the highest
paid people in tech right now. And we're
talking, you know, 150 to 200k plus just
for starting roles. So with that said,
let's dive in and let me discuss how to
learn machine learning like a genius so
that you don't give up. Now I want to
start by discussing the trap that almost
everybody falls into when they're trying
to get into machine learning. Now that's
trying